# Web Scrapping de La Jornada (TEST)

En este notebook se va a realizar un código capaz de extraer información de la página web (pública) del periódico ***La Jornada*** mediante la técnica conocida como **Web Scrapping** con el objetivo de recabar datos sobre eventos que puedan ser catalogados como *masacres*, los cuales deben de contar con al menos 4 cadáveres u homicidios en el mismo evento.

La extracción de información se hará sobre periódicos del **01 de Enero del 2013** al **31 de Diciembre del 2015** los cuales tienen una periodicidad diaria, salvo en fechas especiales. En la Web se tiene el registro de la Portada y Contraportada del periódico, así como el hipervínculo hacia los artículos y las noticias que aparecen en dicha portada y contraportada. Solo se tomarán en cuenta aquellas noticias y artículos que aparezcan en tales hipervínculos pues no se tiene otra forma de acceder al periódico completo. Esto, en principio, no sesga la información ni merma la obtención de datos pues los datos que se quieren extraer hacen referencia a hechos y eventos de violencia extrema, que tienen un alto impacto en la comunidad, y por lo tanto suelen aparecer en alguna de las portadas.

<div style="text-align: center; font-size: 1.5em; font-weight: normal;">
    <b>Este notebook es la aplicación del modelo creado en el notebook anterior</b> (<code>LaJornada_TEST.ipynb</code>) .<br>
    Entradas: Número de días a analizar desde el 01/01/2013 (Se usan 1096 pues son los necesarios hasta el 31/12/2015)<br>
    Salidas: Noticias clasificadas como masacre en formato de valores separados por comas (<code>.csv</code>)
</div>


## Parte 1: Acceso a la página web y Extracción de las URL

Para lograr la recopilación de información, primero se debe acceder de cierta forma a la página web, por lo que se utiliza la librería `request` de python, capaz de enviar peticiones tipo `GET` y `POST`. 

Luego, se usa la librería `bs4` capaz de adaptar y filtrar el formato `HTML` que devuelve la petición anterior y así poder extraer la información en texto plano.

En el flujo de trabajo, primero se extraen todas las **URL** junto al título de todas las noticias que aparecen en todas las portadas y contraportadas dentro de las fechas establecidas. Luego, se comienza con un proceso de filtrado de notas: debido a que se extraen todas las noticias, se incluyen las deportivas, económicas, de espectáculos, internacionales, entre otras secciones que no resultan de interés para el presente estudio; por lo que se pasa por un proceso de filtrado que solo mantiene aquellas noticias que pertenecen a las secciones:

- Política
- Sociedad
- Estados
- Capital

(Tarda aproximadamente 20min para 600 días)

In [4]:
# =========================================
#        Parte 1 y 2: Extraer URL's
# =========================================


# +++++ CODIGO "LEVE", SOLO ENVÍA UN REQUEST POR PORTADA (TOTAL 1095) +++++


import requests # Peticiones get y post
from bs4 import BeautifulSoup # Manejo de html
from datetime import date, timedelta # Manejo de fechas
import lxml
import json
import re # Expresiones regulares
import time # Para ver cuanto tarda
from tqdm.notebook import tqdm # Barra de carga

t1 = time.time()

url_inicial =  "https://web.jornada.com.mx/" 
# A esta URL hay que agregarle al final AAAA/MM/DD para ver la portada y contraportada de esa fecha

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'es-MX,es;q=0.9,en;q=0.8',
    'Referer': 'https://www.google.com.mx/'
} # Los encabezados que "disfrazan" el script de python como si fuera un usuario de chrome

lista_urls = set() # Aqui se inicia la variable para guardar los URL's UNICOS

secciones_de_interes = ['politica',
                        'sociedad',
                        'estados',
                        'capital'
                       ]


# Hay 365 dias por año (el rango de fechas no incluye biciestos), y se quieren ver 3 años = 1095 días
# Se construye una función que indica la fecha tras sumar x días desde el 01/01/2013:
def fecha(i: int):
    '''Devuelve el string de la fecha que resulta de sumar i días al 2013/01/01 (AAAA/MM/DD)'''
    fecha_inicial = date.fromisoformat("2013-01-01")
    fecha_fin = fecha_inicial + timedelta(days = i)
    return re.sub(r"-","/",str(fecha_fin))


def guardar_progreso(archivo: str, progreso: dict):
    with open(archivo, "a", encoding = "utf-8") as file:
        file.write(json.dumps(progreso, ensure_ascii=False) + "\n")


n_dias = 1096
n_url = 0 # Contador de chequeo, se puede quitar después
n_url_util = 0 # Contador de chequeo, se puede quitar después     
urls_diarios = [url_inicial]*n_dias # Aquí se va a guardar la lista de todos los URL
for i in range(n_dias):
    urls_diarios[i] = urls_diarios[i] + fecha(i)

pbar = tqdm(urls_diarios, desc = "Extrayendo URL's", unit = "dias")
contador_i = 0
for url_i in pbar:
    respuesta = requests.get(url_i, headers = headers, timeout = 8) # Hacer petición
    
    # Extrae el HTML
    if str(respuesta.status_code)[0] == '2': # Si el estatus es Exitoso...
        soup = BeautifulSoup(respuesta.text, 'html.parser')
    
        # Primero, traer todas las secciones donde hay noticias (Algunas con hipervinculos)
        lista_divs = soup.find_all("div", class_ = re.compile(r"sumario blanco p16|cabeza|aviso"))
        pbar.set_description(f"Buscando en la fecha: {str(url_i)[27:]}")
        # Extraer todas las URL de los divs
        for div in lista_divs:
            try:
                # Intentar sacar el URL (Algunos no tienen)
                a = div.find('a')
                href = str(a.get('href'))
                section = re.search(r"([^/]+)/", href)
                section = section.group(1)
                n_url += 1
                
                if section in secciones_de_interes: # Filtrar URL's
                    n_url_util += 1
                    if url_i+'/'+href not in lista_urls:
                        guardar_progreso("lista_URL.jsonl", {"fecha": fecha(contador_i),
                                                             "URL"  : url_i + "/" + href})
                        lista_urls.add(url_i + "/" + href )
                
                pbar.set_postfix({"URL's totales":n_url,"URL's post-filtro":n_url_util})
                

            except AttributeError:
                continue
        time.sleep(0.5)

    # Luego, en la sección de abajo donde hay 2 o 3 noticias por sección, se extraen de un XML:
    url_xml = url_i + '/dir.xml'
    respuesta_xml = requests.get(url_xml, headers = headers, timeout = 10)
    if str(respuesta_xml.status_code)[0] == '2': # Si el estatus es Existoso...
        soup_xml = BeautifulSoup(respuesta_xml.content, 'lxml-xml') # Recoger el html
        
        # Primero, traer todas las secciones donde hay noticias (Algunas con hipervinculos)
        lista_items = soup_xml.find_all("Item")
        
        # Extraer las URL de los Items
        for item in lista_items:
            try:
                url_id = str(item.get('id'))
                url_section = str(item.get('section'))
                n_url += 1
                if url_section in secciones_de_interes: # Filtrar URL's
                    n_url_util += 1
                    if url_i + "/" + url_section + "/" + url_id not in lista_urls:
                        guardar_progreso("lista_URL.jsonl", {"fecha": fecha(contador_i),
                                                             "URL"  : url_i + "/" + url_section + "/" + url_id })
                        lista_urls.append( url_i + "/" + url_section + "/" + url_id )
            except Exception as e:
                continue
        time.sleep(0.4)
    contador_i += 1
        
print(f"\n\nEl programa tardó: {(int(t2-t1)//60)//60:3d} horas {(int(t2-t1)//60)%60:3d} min {(t2-t1)%60:6.3f} seg")
print(f"\nSe encontraron {n_url:8d} URL's en {n_dias:5d} periódicos, de las cuales tras filtrar por secciones solo se mantienen {n_url_util:6d} URL's")

Extrayendo URL's:   0%|          | 0/1096 [00:00<?, ?dias/s]



El programa tardó:  2402.0070 segundos

Se encontraron   151600 URL's en  1096 periódicos, de las cuales tras filtrar por secciones solo se mantienen  85504 URL's


## Parte 2: Extracción de información

De las URL's anteriormente extraidas,se extrae el texto de la noticia para poder filtrarlo y analizarlo posteriormente.

Para los 1096 días (76k URL) tarda alrededor de 21 horas. **Es el paso mas tardado**

In [5]:
# =====================================
#       Parte 2: Extraer Texto      
# =====================================


# +++++ CÓDIGO PESADO, ENVÍA UN REQUEST POR *NOTICIA* (TOTAL APROX = 80,000) +++++


import re
import time
import json

t1 = time.time()

def guardar_progreso(archivo: str, progreso: dict):
    with open(archivo, "a", encoding = "utf-8") as file:
        file.write(json.dumps(progreso, ensure_ascii=False) + "\n")

import requests
from bs4 import BeautifulSoup
import re
import random
from tqdm.notebook import tqdm

with open('lista_URL.jsonl', 'r') as lista_url:
    total_url = sum(1 for _ in lista_url)

with open('lista_URL.jsonl', 'r') as lista_url:
    url_procesadas = set()
    pbar = tqdm(lista_url, total = total_url, desc="Intentando scraping", unit="url")
    for linea in pbar:
        linea_py = json.loads(linea)
        url = linea_py["URL"]
        fecha = linea_py["fecha"]
        
        try:
            time.sleep(abs(random.gauss(.3,0.05))) # Esperamos un poco para no saturar el servidor
            
            # Descarga con timeout para que no se trabe el script
            respuesta = requests.get(url, headers = headers, timeout=10)
            
            # Procesamos el texto completo de una vez
            sopa = BeautifulSoup(respuesta.content, 'html.parser')
            
            # Buscamos en todo el artículo (Ignoramos encabezado, menús, publicidad, etc)
            art = sopa.find("div", id = "article-text")
            
            texto_completo = art.get_text(separator=' ', strip=True)

            guardar_progreso("texto_noticias.jsonl", {"fecha": fecha,
                                                      "URL": url,
                                                      "texto": texto_completo})

        except Exception as e:
            print(f"Error en {url}:\t{e}")
            continue



t2 = time.time()
print(f"\nEl programá se ejecutó durante {(int(t2-t1)//60)//60:3d} horas {(int(t2-t1)//60)%60:3d} min {(t2-t1)%60:6.3f} seg")


Intentando scraping: 0url [00:00, ?url/s]

Error en https://web.jornada.com.mx/2013/04/25/estados/041n7est:	HTTPSConnectionPool(host='web.jornada.com.mx', port=443): Max retries exceeded with url: /2013/04/25/estados/041n7est (Caused by NewConnectionError("HTTPSConnection(host='web.jornada.com.mx', port=443): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
Error en https://web.jornada.com.mx/2013/04/25/estados/041n8est:	HTTPSConnectionPool(host='web.jornada.com.mx', port=443): Max retries exceeded with url: /2013/04/25/estados/041n8est (Caused by NewConnectionError("HTTPSConnection(host='web.jornada.com.mx', port=443): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
Error en https://web.jornada.com.mx/2013/05/08/politica/005n1pol:	HTTPSConnectionPool(host='web.jornada.com.mx', port=443): Max retries exceeded with 

## Parte 3: Filtrar noticias relevantes

Solo se mantienen las URL tales que en el contenido de la noticia (ya sea en el título, pies de página o los párrafos) contenga alguna de las siguientes ***palabras clave*** o sus variantes (plurales, singulares, masculinos, femeninos, conjugaciones).

- Masacre(s), masacraron, masacrado(s)
- Matanza(s)
- Multihomicidio(s)
- Plurihomicidio(s)
- Exterminio(s), exterminad(os-as), exterminación
- Asesinato(s), asesinad(os-as), asesin(os-as)
- Cadáver(es)
- Homicidio(s), homicida
- Muerte(s), muerto(s), mueren
- Murieron
- Matar, mataron
- Sin vida
- Perdieron la vida
- Agresión, agresores, agredidos, agredió, agredieron
- Balacera
- Tiroteo(s), tirotearon
- Disparó, Dispararon, Disparad(os-as)
- Defunción, defunciones
- Fallecido, fallecida, fallecieron, falleció
- Feminicidio, feminicidios, feminicida
- Torturad(os-as), Tortura(s)
- Maniatad(os-as) = Con manos atadas
- Decapitad(os-as)
- Desmembrad(os-as)
- Ejecucion(es), ejecutó, ejecutad(os-as)
- Abatid(os-as), Abatieron, Abaten, Abatió



In [19]:
# =====================================
#    Parte 3: Filtrar por Keyword      
# =====================================


import re
import time
import json

t1 = time.time()

def guardar_progreso(archivo: str, progreso: dict):
    with open(archivo, "a", encoding = "utf-8") as file:
        file.write(json.dumps(progreso, ensure_ascii=False) + "\n")

# Definimos las raíces para cubrir plurales y variaciones
# \b al inicio y final asegura que busque palabras completas y no partes de otras
patrones = [
    r"masacr\w+",                  # Masacre(s), masacraron, masacrado(s)
    r"matanz\w+",                  # Matanza(s)
    r"multihomicidio\w*",          # Multihomicidio(s)
    r"plurihomicidio\w*",          # Plurihomicidio(s)
    r"extermin\w+",                # Exterminio(s), exterminad(os-as), exterminación
    r"asesin\w+",                  # Asesinato(s), asesinad(os-as), asesin(os-as)
    r"cad[aá]ver\w*",              # Cadáver(es)
    r"homicid\w+",                 # Homicidio(s), homicida
    r"muer\w+",                    # Muerte(s), muerto(s), mueren
    r"murieron",                   # Palabra exacta
    r"matar\w*",                   # Matar, mataron
    r"sin vida",                   # Frase exacta
    r"perdieron la vida",          # Frase exacta
    r"agres\w+",                   # Agresión, agresores, agredidos, agredió, agredieron
    r"balacera\w*",                # Palabra exacta
    r"tirote\w+",                  # Tiroteo(s), tirotearon
    r"dispar\w+",                  # Disparó, Dispararon, Disparad(os-as)
    r"defunci\w+",                 # Defunción, defunciones
    r"falleci\w+",                 # Fallecido, fallecida, fallecieron, falleció
    r"feminicid\w+",               # Feminicidio, feminicidios, feminicida
    r"tortur\w+",                  # Torturad(os-as), Tortura(s)
    r"maniatad\w+",                # Maniatad(os-as) = Con manos atadas
    r"decapita\w+",                # Decapitad(os-as)
    r"desmembrad\w+",              # Desmembrad(os-as)
    r"\bejecu(tó|taron|tados?|tadas?|ci[oó]n(es)?)\w+", # Para evitar "Ejecutivo o Ejecutiva" pero añadir "Ejecutados" o "ejecuciones"
    r"abati\w+"                    # Abatid(os-as), Abatieron, Abaten, Abatió
    
]

# Unimos todo con el operador OR (|)
regex = re.compile("|".join(patrones), re.IGNORECASE)

import re
from tqdm.notebook import tqdm
import json

# Tu lista de URLs únicas (usa set() para asegurar que no haya repetidas)

url_procesadas = set()

with open('texto_noticias.jsonl', 'r', encoding = "utf-8") as noticias:
    total_noticias = sum(1 for _ in noticias)

with open('texto_noticias.jsonl', 'r', encoding = "utf-8") as noticias:
    pbar = tqdm(noticias, total=total_noticias, desc="Buscando Matches", unit="noticias")
    for linea in pbar:
        noticia = json.loads(linea)
        url = noticia["URL"]
        fecha = noticia["fecha"]
        texto_completo = noticia["texto"]

        # Buscamos el match
        match = regex.search(texto_completo)
    
        if match:
            # Guardamos un diccionario de la noticia
            noticia_match = {
                "fecha": fecha,
                'URL': url,
                'keyword': match.group(),
                'texto': texto_completo 
            }
            # Si encontramos algo, guardamos y salimos del loop de esta URL
            # Guardamos el match.group() para saber cual palabra activó la alerta
    
            if url not in url_procesadas:
                url_procesadas.add(url)
                guardar_progreso("noticias_con_match.jsonl", noticia_match)
                     
            # Actualizamos las estadísticas (a la derecha de la barra)
            pbar.set_postfix({"Matchs": len(url_procesadas)})
    

t2 = time.time()
print(f"\nEl programá se ejecutó durante {(int(t2-t1)//60)//60:3d} horas {(int(t2-t1)//60)%60:3d} min {(t2-t1)%60:6.3f} seg")

print(f"\nSe Obtuvieron {len(url_procesadas)} noticias con coincidencias de alguna palabra clave, y se guardaron para su posterior análisis")

Buscando Matches:   0%|          | 0/76680 [00:00<?, ?noticias/s]


El programá se ejecutó durante   0 horas   1 min  4.840 seg

Se Obtuvieron 20660 noticias con coincidencias de alguna palabra clave, y se guardaron para su posterior análisis


## Parte 4: Análisis de lenguaje natural

Tras filtrar solo aquellas noticias que tengan coincidencias con alguna de las palabras clave, se requiere poder analizar de manera precisa y eficiente el contenido de la noticia para identificar el número de víctimas, el lugar de ocurrencia, la fecha y alguna otra característica del evento que se desee analizar. Para esto, se utilizará la librería `spaCy` en su módulo de `es_core_news_lg`, el cual es la versión más grande, y por lo tanto, potente y precisa para procesar texto escrito en español y categorizarlo (Ubicación, Personajes, Tiempo, etc.).

Tras este análisis de lenguaje natural (NLP), se guardarán los datos en un archivo para una comprobación humana de una muestra, el guardado se hará individualmente tras cada noticia. Esta es una técnica de *programación defensiva* y permite una mayor seguridad ante bloqueos de la página web, o errores del código.

In [ ]:
# ===========================
#        Limpiar datos
# ===========================

from IPython.display import clear_output
import pandas as pd
from random import choice
import time
import re

t1 = time.time()

# Definimos las palabras que se quieran asociar al número de victimas
keywords = [r"muert[oa]s",
            r"víctimas",
            r"cuerpos",
            r"occisos",
            r"fallecid[oa]s",
            r"cadáver(?:es)",
            r"restos? humanos?",
            r"ejecutad[oa]s?",
            r"abatid[oa]s?", 
            r"decesos?",
            r"homicidios?",
            r"asesinatos?"]
string_kw = "|".join(keywords) # Aqui se unen en una string separados por "|"

# El número de víctimas podría estar con número o con letras (esta expresion regular solo acepta hasta el número 20 en letras)
numeros = [r"un[oa]?",
          r"dos",
          r"tres",
          r"cuatro",
          r"cinco",
          r"seis",
          r"siete",
          r"ocho",
          r"nueve",
          r"diez",
          r"once",
          r"doce",
          r"trece",
          r"catorce",
          r"quince",
          r"dieciséis",
          r"diecisiete",
          r"dieciocho",
          r"diecinueve",
          r"veinte",
          r"\d+"]
string_num = "|".join(numeros)

texto_a_num = {
    "un"        : 1, "una": 1, "uno": 1,
    "dos"       : 2,
    "tres"      : 3,
    "cuatro"    : 4,
    "cinco"     : 5,    
    "seis"      : 6,
    "siete"     : 7,
    "ocho"      : 8,
    "nueve"     : 9,
    "diez"      : 10,
    "once"      : 11,
    "doce"      : 12,
    "trece"     : 13,
    "catorce"   : 14,
    "quince"    : 15,
    "dieciséis" : 16,
    "diecisiete": 17,
    "dieciocho" : 18,
    "diecinueve": 19,
    "veinte"    : 20
}

# La regex final (usando el puente de 0 a 3 palabras)
patron_victimas = re.compile(
    rf"\b(?:({string_num})\s+(?:[\w,\.]+\s+){{0,5}}(?:{string_kw})|(?:{string_kw})\s+(?:[\w,\.]+\s+){{0,5}}({string_num}))\b",
    flags=re.IGNORECASE | re.UNICODE
)

# Función auxiliar para normalizar llaves
from unidecode import unidecode
def normalizar(texto):
    if isinstance(texto, str):
        return unidecode(str(texto)).lower().strip()
    if isinstance(texto, list):
        a=[]
        for palabra in texto:
            a.append(unidecode(str(palabra)).lower().strip())
        return a



df_cat = pd.read_excel('CATALOGO_MUN_PAIS.xlsx')

dict_municipios = {normalizar(k): v for k, v in zip(df_cat['NOMBRE_MUN'], df_cat['CVEGEO'])}

# 2. Diccionario de Estados (usamos tanto el nombre como la abreviatura como llaves)
dict_estados = {normalizar(k): v for k, v in zip(df_cat['NOMBRE_ENT'], df_cat['CVE_ENT'])}
dict_estados.update({normalizar(k): v for k, v in zip(df_cat['ABR_ENT'], df_cat['CVE_ENT'])})

dict_abr = {k: normalizar(v) for k, v in zip(df_cat["CVE_ENT"], df_cat["ABR_ENT"])}
dict_label = {normalizar(k): v for k, v in zip(df_cat["LABEL_MUN"], df_cat["CVEGEO"])}


# Asumiendo que df_geo tiene: municipio_norm, estado_norm
dict_maestro_ = df_cat.groupby('NOMBRE_MUN')['NOMBRE_ENT'].apply(list).to_dict()
dict_maestro = {normalizar(k): normalizar(v) for k, v in dict_maestro.items()}




import spacy

# 1. Carga el modelo de español (usa el 'lg' si quieres máxima precisión)
# Desactivamos lo que no usaremos para ganar velocidad
nlp = spacy.load("es_core_news_lg", disable=["parser", "lemmatizer", "textcat"])

def extraer_localizaciones(texto):
    if not (isinstance(texto, str)) or texto.strip() == "":
        return None, "No texto"
    
    doc = nlp(texto)
    locs_raw = [ent.text for ent in doc.ents if ent.label_ in ["LOC", "GPE"]]
    locs_norm = [normalizar(loc) for loc in set(locs_raw)]
    
    if not locs_norm:
        return None, "No locs"

    # 1. Identificar qué estados se mencionan en la noticia (Contexto)
    estados_mencionados = {normalizar(l) for l in locs_norm if normalizar(l) in dict_estados}
    
    # 2. Buscar municipios
    for loc in locs_norm:
        if len(loc) < 4: continue  # Seguridad: evita que "al", "la" o "san" hagan match
        
        # a. Intento de Match Exacto (Eficiencia O(1))
        if loc in dict_municipios:
            return dict_municipios[loc], "Municipio (Exacto)"
        
        # b. Intento de "LIKE" SQL (Contención)
        # Buscamos si 'acapulco' está contenido en 'acapulco de juarez'
        match_like = [cve for nombre, cve in dict_municipios.items() if loc in nombre]
        
        if match_like:
            # Si hay varios, tomamos el primero (o podrías aplicar más lógica)
            return match_like[0], "Municipio (LIKE)"

    # 3. Si no hubo municipio, buscamos si al menos hay un estado
    if estados_mencionados:
        # Tomamos el primero de los estados mencionados
        edo_unico = list(estados_mencionados)[0]
        return dict_estados[edo_unico], "Estado"

    return None, "No locs o matches"


def procesar_num_victimas(noticia : str):
    
    match_victimas = re.search(patron_victimas, noticia)

    if match_victimas:
        # Guardamos el match (el número de victimas)
        victimas_re = match_victimas.group(1) or match_victimas.group(2)
        victimas_re = victimas_re.lower().strip()
        
        # El Match podría ser el texto "siete" o el número 4. Pasamos todo a int
        if victimas_re.isdigit():
            victimas_re = int(victimas_re)
        else:
            victimas_re = texto_a_num.get(victimas_re, 1)
        resultado_final = victimas_re
    else: 
        # Si no hay match de RegEx, guardamos la noticia aparte (codigo 0)
        resultado_final = 0
            
    return resultado_final

# Cargar datos para etiquetar
df_noticias = pd.read_json("noticias_con_match.jsonl", lines = True)

def limpiar_minimo(texto):
    texto = texto.lower()
    # Eliminar números de patrullas, fechas o años (4 dígitos) que confunden
    texto = re.sub(r'\b\d{4}\b', '', texto) 
    # Eliminar puntuación básica
    texto = re.sub(r'[^\w\s]', '', texto)
    # Eliminar acentos (peligroso) pero util
    texto = unidecode(str(texto)).lower().strip()
    return texto

df_noticias['texto_limpio'] = df_noticias['texto'].apply(limpiar_minimo)

# Crear columna
df_noticias["num_vict_regex"] = df_noticias["texto_limpio"].apply(procesar_num_victimas)

df_noticias[["localizacion", "tipo_loc"]] = df_noticias["texto_limpio"].apply(lambda x: pd.Series(extraer_localizaciones(x)))


noticias_to_csv = df_noticias[["fecha", "URL", "texto_limpio", "num_vict_regex", "localizacion", "tipo_loc"]]

noticias_to_csv.to_csv("noticias_texto_limpio.csv", index = False)

t2 = time.time()
print(f"\nEl programá se ejecutó durante {(int(t2-t1)//60)//60:3d} horas {(int(t2-t1)//60)%60:3d} min {(t2-t1)%60:6.3f} seg")

### Utilizar el Modelo de Clasificación (ML)

Con la librería `scikit-learn` se entrenó a un modelo de Clasificación (Aprendizaje Supervisado) con aproximadamente 400 datos (+200 clasificados como masacre). Se repite el proceso un total de 300 veces, generando un modelo nuevo en cada ocasión para verificar la estabilidad, confiabilidad y robustez del mismo ante un cambio en las muestras tanto de entrenamiento como de prueba.

El modelo resultante obtuvo una precisión superior al 80% y un recall superior al 90%, con un f1-score aproximado de 0.85. Aqui se utiliza para clasificar si una noticia trata sobre masacres (binario).

In [48]:
import joblib
import time

t1 = time.time()
# Cargar el cerebro que ya entrenaste
model = joblib.load('modelo_masacres_final.pkl')
vectorizer = joblib.load('vectorizador_final.pkl')

# Cargar tus 80k-95k noticias ya scrappeadas
df_total = pd.read_csv('noticias_texto_limpio.csv')

# Vectorizar y predecir
X_total = vectorizer.transform(df_total['texto_limpio'])
df_total['prediccion_ML'] = model.predict(X_total)
# Importante: Obtener la confianza para filtrar casos dudosos
df_total['confianza_ML'] = model.decision_function(X_total)

df_total.to_csv("predicciones_ML.csv", index = False)

t2 = time.time()

print(f"El código tardó {t2-t1: 6.2f}seg")

El código tardó  13.1661seg


In [41]:
#dict_municipios
#dict_estados
dict_abr
#dict_label
#dict_maestro

{1: 'ags',
 2: 'bc',
 3: 'bcs',
 4: 'camp',
 5: 'coah',
 6: 'col',
 7: 'chis',
 8: 'chih',
 9: 'cdmx',
 10: 'dgo',
 11: 'gto',
 12: 'gro',
 13: 'hgo',
 14: 'jal',
 15: 'mex',
 16: 'mich',
 17: 'mor',
 18: 'nay',
 19: 'nl',
 20: 'oax',
 21: 'pue',
 22: 'qro',
 23: 'qroo',
 24: 'slp',
 25: 'sin',
 26: 'son',
 27: 'tab',
 28: 'tamps',
 29: 'tlax',
 30: 'ver',
 31: 'yuc',
 32: 'zac'}